# Phase 4 CJPE - Model 3 (50k BPE, MTP-pretrained) + **LoRMA+** + attention pooling across chunks
Run the cells top to bottom. **First run once with `SMOKE = True`** (~5 min) to verify paths/tokenizer/checkpoint, then set it to `False`.

Attach as Kaggle inputs: (1) the Model-3 checkpoint dataset (`best.pt` + `best_meta.json`), (2) the tokenizer files the Model-3 run used (`vocab_50k.json` + `merges_50k.json`, and/or an HF `tokenizer.json` with a 50k vocab). Kaggle secret `HF` = your Hugging Face token.

Design: frozen Model-3 backbone + **LoRMA+** adapters (multiplicative update `y = (I + a/r * B A) W0 x`, Bihany et al., ACL Findings 2025) on all attention + SwiGLU projections -> per-window features -> **gated-attention pooling across overlapping 1024-token chunks** (+ chunk-position embeddings, tail branch, auxiliary per-chunk loss) -> binary head. Class-weighted CE, threshold tuned on leak-free dev.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
import re, gc, sys, json, math, time, random, shutil, zipfile, hashlib, warnings
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")
T0 = time.time()

# ------------------------------------------------------------------ USER SETTINGS
SMOKE            = False          # True = ~5 min dry run on a tiny subset. RUN THIS ONCE FIRST.
MODE             = "train_predict"  # "train_predict" | "predict_only" (needs best_adapter_m3.pt from an earlier run)
TRAIN_BUDGET_MIN = 360            # total training minutes (LR schedule is time-based, so it always anneals to the end)
MAX_EPOCHS       = 3              # Model-2 peaked at ~epoch 3 of 5, so 3 is enough
SESSION_LIMIT_H  = 11.0           # stop training in time to still run inference before Kaggle's 12h wall
RESERVE_MIN      = 25             # minutes kept free for dev + test inference

CKPT_PATH        = None           # optional: "/kaggle/input/<model3-dataset>/.../best.pt"
TOKENIZER_DIR    = None           # optional: directory holding the tokenizer files
TOKENIZER_CHOICE = "auto"         # "auto" (probe all found tokenizers on the model's LM loss) | "legalbpe" | "hf"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_FILES = {"train": "Phase_4_CJPE/Train/train.json",
                "dev":   "Phase_4_CJPE/Train/dev.json",
                "test":  "Phase_4_CJPE/Test/test.json"}
LOWERCASE = True                  # dev/test are lowercased (train is mixed-case) -> match the test distribution
EXCLUDE_TEST_IDS_FROM_DEV = True  # every test ID also sits in dev.json; keep them out of model selection / threshold tuning
MAX_CHARS, HEAD_CHARS, TAIL_CHARS = 400_000, 80_000, 240_000

# overlapping windows (same geometry Model-3 was pretrained with: 1024 ctx, stride 768)
CHUNK, STRIDE = 1024, 768
K_TRAIN, K_QUICK, MAX_INFER_WINDOWS = 6, 8, 40
NB = 8                            # relative-position buckets (+ first + last)

# adapter: LoRMA+ (compulsory for this phase)
ADAPTER = "lorma_plus"            # "lorma_plus" (multiplicative, I + s*BA applied on W0 x) | "lora" (additive baseline)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down")
BACKBONE_DROPOUT = 0.1            # = Model-3 pretraining dropout

# optimisation
MICRO_DOCS, ACCUM = 2, 8          # 2 docs x 6 windows per micro-batch; 16 docs / optimizer step
LR_LORA, LR_HEAD, WD_HEAD = 3e-4, 1e-3, 0.01
WARMUP_STEPS, MIN_LR_RATIO, GRAD_CLIP = 50, 0.05, 1.0
AUX_W = 0.3                       # weight of the per-window auxiliary loss
LOG_EVERY, EVAL_EVERY_MIN, CKPT_EVERY_MIN = 20, 30.0, 15.0
N_QUICK = 400
INFER_BS, DOC_BS = 32, 64
SEED = 1234

if SMOKE:
    TRAIN_BUDGET_MIN, MAX_EPOCHS, RESERVE_MIN = 3, 1, 0
    EVAL_EVERY_MIN, CKPT_EVERY_MIN, N_QUICK, MAX_INFER_WINDOWS = 1.0, 1.0, 40, 8
    N_SMOKE = {"train": 160, "dev": 90, "test": 40}

# ------------------------------------------------------------------ paths / device
IS_KAGGLE  = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
WORK = Path("/kaggle/working" if IS_KAGGLE else ".") / ("phase4_model3_lorma" + ("_smoke" if SMOKE else ""))
CACHE, CKPT, OUT, SUB = WORK / "cache", WORK / "ckpt", WORK / "out", WORK / "submission"
for _d in (CACHE, CKPT, OUT, SUB):
    _d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
OOM_ERR = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)

def amp_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP)

def log(*a):
    print(*a, flush=True)

def fmt_t(sec):
    sec = int(max(sec, 0)); return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

def read_json(path):
    try:
        with open(path, encoding="utf-8") as f: return json.load(f)
    except Exception:
        return None

def find_files(name, roots):
    hits = []
    for r in roots:
        if r and os.path.isdir(r):
            for d, _, fs in os.walk(r):
                if name in fs: hits.append(os.path.join(d, name))
    return hits

def atomic_save(obj, path):
    tmp = str(path) + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

def _hf_token():
    t = os.environ.get("HF_TOKEN")
    if t: return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")      # never hard-code the token in a notebook
    except Exception:
        return None

# ------------------------------------------------------------------ data access + text cleaning
def _download(split):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE4_FILES[split], repo_type="dataset", token=_hf_token())

def _records(split):
    obj = read_json(_download(split))
    if isinstance(obj, dict):
        for k in ("data", "records", "examples"):
            if isinstance(obj.get(k), list): obj = obj[k]; break
    return obj

_ZW, _CTRL = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]"), re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_HYPH, _WRAP = re.compile(r"(?<=\w)-\n(?=\w)"), re.compile(r"(?<=\w)\n(?=\w)")
_SP, _NLSP, _NL3 = re.compile(r"[ \t]+"), re.compile(r"[ \t]*\n[ \t]*"), re.compile(r"\n{3,}")

def prep_text(t):
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = _CTRL.sub(" ", _ZW.sub("", t)).replace("\ufffd", " ")
    t = _HYPH.sub("", t)
    t = _WRAP.sub(" ", t)                      # undo hard line-wraps (dev/test are wrapped at ~50 chars)
    t = _NL3.sub("\n\n", _NLSP.sub("\n", _SP.sub(" ", t))).strip()
    if len(t) > MAX_CHARS:                     # keep the beginning and (mostly) the end of giant judgments
        t = t[:HEAD_CHARS] + "\n" + t[-TAIL_CHARS:]
    return t.lower() if LOWERCASE else t

log(f"device={DEVICE} | amp={USE_AMP} ({AMP_DTYPE}) | SMOKE={SMOKE} | MODE={MODE} | adapter={ADAPTER} | work dir={WORK}")
if DEVICE.type == "cuda":
    log("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = 50048
    real_vocab: int = 50000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 1536
    dropout: float = 0.1
    max_len: int = 1088
    mtp_heads: int = 3

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        inv_rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)
        return x * inv_rms * self.weight.to(x.dtype)

def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):
    cos, sin = cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

def repeat_kv(x, groups):
    if groups == 1: return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)

class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.kv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)
    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up   = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.attn = RMSNorm(cfg.d_model), GQAttention(cfg)
        self.norm2, self.ffn = RMSNorm(cfg.d_model), SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        return x + self.drop(self.ffn(self.norm2(x)))

class MTPHead(nn.Module):
    """z = RMSNorm(x + W x). Head 0 is the ordinary next-token head and acts as the model's final norm."""
    def __init__(self, dim):
        super().__init__(); self.proj = nn.Linear(dim, dim, bias=False); self.norm = RMSNorm(dim)
    def forward(self, x):
        return self.norm(x + self.proj(x))

class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.heads = nn.ModuleList([MTPHead(cfg.d_model) for _ in range(cfg.mtp_heads)])
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def _rope(self, T):
        return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
    def hidden(self, idx):
        cos, sin = self._rope(idx.size(1))
        x = self.drop(self.embed(idx))
        for blk in self.blocks: x = blk(x, cos, sin)
        return x

def locate_checkpoint():
    if CKPT_PATH: return CKPT_PATH
    best = None
    for h in find_files("best.pt", [INPUT_ROOT]):
        meta = read_json(os.path.join(os.path.dirname(h), "best_meta.json")) or {}
        if "mtp" not in str(meta.get("tag", "")): continue          # skips the Model-1/2 checkpoints
        vl = meta.get("val_loss", 1e9)
        if best is None or vl < best[0]: best = (vl, h)
    if best: log(f"Model-3 best.pt: next-token val_loss={best[0]:.4f}"); return best[1]
    for h in find_files("latest.pt", [INPUT_ROOT]):
        meta = read_json(os.path.join(os.path.dirname(h), "latest_meta.json")) or {}
        if "mtp" in str(meta.get("tag", "")): log("best.pt not found - falling back to latest.pt"); return h
    raise FileNotFoundError("Model-3 (MTP) best.pt / latest.pt not found - attach the Model-3 checkpoint dataset or set CKPT_PATH")

MODEL_CKPT = locate_checkpoint()
log(f"checkpoint: {MODEL_CKPT}")

def load_base_model():
    ck = torch.load(MODEL_CKPT, map_location="cpu", weights_only=False)
    sd = {k.replace("_orig_mod.", "", 1): v for k, v in ck["model"].items()}
    cd = dict(ck.get("config", {}))
    V, D = sd["embed.weight"].shape
    hd = D // cd["n_heads"] if cd.get("n_heads") else 64
    cfg = ModelConfig(
        vocab_size=V, real_vocab=int(cd.get("real_vocab", V)), d_model=D,
        n_layers=1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks.")),
        n_heads=sd["blocks.0.attn.q_proj.weight"].shape[0] // hd,
        n_kv_heads=sd["blocks.0.attn.k_proj.weight"].shape[0] // hd,
        ffn_hidden=sd["blocks.0.ffn.w_gate.weight"].shape[0],
        dropout=BACKBONE_DROPOUT, max_len=int(cd.get("max_len", 1088)),
        mtp_heads=sum(1 for k in sd if k.startswith("heads.") and k.endswith(".proj.weight")))
    for k in ("d_model", "n_layers", "n_heads", "n_kv_heads", "ffn_hidden", "mtp_heads"):
        if k in cd and cd[k] != getattr(cfg, k): log(f"  note: saved config says {k}={cd[k]} but the weights say {getattr(cfg, k)} -> using the weights")
    assert CHUNK <= cfg.max_len, "CHUNK exceeds the RoPE table of the checkpoint"
    m = DecoderLM(cfg)
    m.load_state_dict(sd, strict=True)                  # strict: fails loudly if the architecture copy differs
    if "val_loss" in ck: log(f"loaded Model-3 weights (pretraining next-token val loss {ck['val_loss']:.4f}, step {ck.get('global_step')})")
    del ck, sd
    for p in m.parameters(): p.requires_grad = False
    return m.to(DEVICE), cfg

model, MCFG = load_base_model()
log(f"Model-3 backbone: d={MCFG.d_model} layers={MCFG.n_layers} heads={MCFG.n_heads}/{MCFG.n_kv_heads}kv ffn={MCFG.ffn_hidden} "
    f"mtp_heads={MCFG.mtp_heads} embed_rows={MCFG.vocab_size} | {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

WORD_PATTERN = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d|[a-zA-Z]+|[0-9]+|[^\s\a-zA-Z0-9]+")

class LegalBPE:
    name = "legalbpe"
    def __init__(self, vocab_path, merges_path):
        self.vocab = read_json(vocab_path)
        merges = read_json(merges_path)
        h = hashlib.md5()
        for p in (vocab_path, merges_path): h.update(open(p, "rb").read())
        self.fingerprint = h.hexdigest()[:10]
        self.ranks = {(a, b): i for i, (a, b) in enumerate(merges)}
        inv = {int(i): t for t, i in self.vocab.items()}
        self.vocab_size = max(inv) + 1
        assert len(inv) == self.vocab_size, "vocab ids are not contiguous"
        self.unk_id = self.vocab.get("<|unk|>", 1)
        self.cache = {}
    def _word_ids(self, word):
        hit = self.cache.get(word)
        if hit is not None: return hit
        syms = list(word) + ["</w>"]
        while len(syms) > 1:
            best, best_rank = None, None
            for i in range(len(syms) - 1):
                r = self.ranks.get((syms[i], syms[i + 1]))
                if r is not None and (best_rank is None or r < best_rank): best, best_rank = (syms[i], syms[i + 1]), r
            if best is None: break
            out, i = [], 0
            while i < len(syms):
                if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]: out.append(best[0] + best[1]); i += 2
                else: out.append(syms[i]); i += 1
            syms = out
        ids = [self.vocab.get(s, self.unk_id) for s in syms]
        if len(self.cache) > 3_000_000: self.cache.clear()
        self.cache[word] = ids
        return ids
    def encode(self, text):
        ids = []
        for w in WORD_PATTERN.findall(text.lower()): ids.extend(self._word_ids(w))
        return ids
    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]

class HFTok:
    name = "hf"
    def __init__(self, path):
        from tokenizers import Tokenizer
        self.t = Tokenizer.from_file(path); self.t.no_truncation(); self.t.no_padding()
        self.vocab_size = self.t.get_vocab_size()
        self.fingerprint = hashlib.md5(open(path, "rb").read()).hexdigest()[:10]
    def encode_batch(self, texts):
        return [e.ids for e in self.t.encode_batch(texts, add_special_tokens=False)]

def find_tokenizers():
    roots = [TOKENIZER_DIR, INPUT_ROOT, str(WORK), os.getcwd()]
    fits = lambda n: MCFG.vocab_size - 64 < n <= MCFG.vocab_size          # real vocab must match the model's (padded) embedding
    found = {}
    if TOKENIZER_CHOICE in ("auto", "legalbpe"):
        for v in find_files("vocab_50k.json", roots):
            m = os.path.join(os.path.dirname(v), "merges_50k.json")
            if os.path.exists(m):
                try:
                    t = LegalBPE(v, m)
                    if fits(t.vocab_size): found["legalbpe"] = t; log(f"tokenizer candidate legalbpe: {v} (vocab {t.vocab_size})"); break
                except Exception as e:
                    log(f"could not load {v}: {e}")
    if TOKENIZER_CHOICE in ("auto", "hf"):
        for h in find_files("tokenizer.json", roots):
            try:
                t = HFTok(h)
                if fits(t.vocab_size): found["hf"] = t; log(f"tokenizer candidate hf: {h} (vocab {t.vocab_size})"); break
            except Exception:
                continue
    if not found: raise FileNotFoundError("no 50k tokenizer found: attach vocab_50k.json+merges_50k.json and/or a 50k tokenizer.json (or set TOKENIZER_DIR)")
    return found

@torch.no_grad()
def lm_probe(tok, texts, n_tok=512):
    """mean next-token CE of the pretrained model on CJPE text under this tokenizer (right tokenizer ~3-4 nats, wrong one >7)"""
    model.eval(); tot = cnt = 0
    for t in texts:
        ids = tok.encode_batch([t])[0][:n_tok + 1]
        if len(ids) < 64: continue
        x = torch.tensor([ids[:-1]], device=DEVICE); y = torch.tensor(ids[1:], device=DEVICE)
        with amp_ctx(): z = model.heads[0](model.hidden(x))
        logits = F.linear(z.float()[0], model.embed.weight.float())
        tot += F.cross_entropy(logits, y, reduction="sum").item(); cnt += y.numel()
    return tot / max(cnt, 1)

CANDS = find_tokenizers()
_probe = [prep_text(r.get("text") or "")[:12000] for r in _records("dev")[:100]]
_probe = [t for t in _probe if len(t) > 3000][:24]
probe_loss = {k: lm_probe(t, _probe) for k, t in CANDS.items()}
for k, v in probe_loss.items(): log(f"  tokenizer probe [{k}]: pretrained-LM next-token loss on {len(_probe)} CJPE docs = {v:.3f}")
TOK = CANDS[min(probe_loss, key=probe_loss.get)]
VOCAB = TOK.vocab_size
if probe_loss[TOK.name] > 6.0: log("!! WARNING: even the best tokenizer gives a very high LM loss -> tokenizer/checkpoint pair is probably mismatched. STOP and check the attached inputs.")
log(f"==> using tokenizer '{TOK.name}' (vocab {VOCAB}, fingerprint {TOK.fingerprint})")
assert VOCAB < 65536, "token ids are cached as uint16"

In [ ]:
class TokDS:
    """tokenised split stored as one flat uint16 array + offsets"""
    def __init__(self, ids, labels, flat, offs, has_upper):
        self.ids, self.labels, self.flat, self.offs, self.has_upper = list(ids), np.asarray(labels), flat, offs, has_upper
        self.n_tok = np.diff(offs)
    def __len__(self): return len(self.ids)
    def slice(self, i, s, l): return self.flat[self.offs[i] + s: self.offs[i] + s + l]

SIG = f"{'smoke' if SMOKE else 'full'}_lc{int(LOWERCASE)}_ex{int(EXCLUDE_TEST_IDS_FROM_DEV)}_{TOK.name}{TOK.fingerprint}_v{VOCAB}"

def _tokenise(texts, bs=128):
    flat, lens, t0 = [], [], time.time()
    for i in range(0, len(texts), bs):
        for ids in TOK.encode_batch(texts[i:i + bs]):
            a = np.asarray(ids if len(ids) else [0], dtype=np.uint16); flat.append(a); lens.append(len(a))   # empty doc -> 1 pad token (no NaN means)
        if (i // bs) % 40 == 0: log(f"    tokenised {min(i + bs, len(texts)):,}/{len(texts):,} ({time.time()-t0:.0f}s)")
    return np.concatenate(flat), np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)

def build_split(split, records, with_labels):
    t0 = time.time()
    ids = [str(r["id"]) for r in records]
    raw = [r.get("text") or "" for r in records]
    has_upper = np.array([t[:5000] != t[:5000].lower() for t in raw])
    labels = np.array([int(r["label"]) for r in records]) if with_labels else np.full(len(records), -1)   # test labels are NEVER read
    flat, offs = _tokenise([prep_text(t) for t in raw])
    log(f"  {split}: {len(ids):,} docs, {len(flat)/1e6:.1f}M tokens in {(time.time()-t0)/60:.1f} min")
    return TokDS(ids, labels, flat, offs, has_upper)

def save_split(split, ds):
    np.savez(CACHE / f"{split}_{SIG}.npz", ids=np.array(ds.ids), labels=ds.labels, flat=ds.flat, offs=ds.offs, has_upper=ds.has_upper)

def load_split(split):
    p = CACHE / f"{split}_{SIG}.npz"
    if not p.exists(): return None
    z = np.load(p)
    return TokDS(z["ids"].tolist(), z["labels"], z["flat"], z["offs"], z["has_upper"])

def get_all_splits():
    cached = {s: load_split(s) for s in ("train", "dev", "test")}
    if all(v is not None for v in cached.values()):
        log("tokenised splits loaded from cache"); return cached
    rng = np.random.default_rng(SEED)
    log("reading CJPE json ...")
    test_rec = _records("test")
    test_ids = {str(r["id"]) for r in test_rec}                     # IDs only - test labels are not touched
    test_rec = [{"id": r["id"], "text": r.get("text")} for r in test_rec]
    dev_rec = _records("dev")
    n_overlap = sum(str(r["id"]) in test_ids for r in dev_rec)
    log(f"!! dev/test ID overlap: {n_overlap:,} of {len(test_ids):,} test IDs are also in dev.json")
    if EXCLUDE_TEST_IDS_FROM_DEV:
        dev_rec = [r for r in dev_rec if str(r["id"]) not in test_ids]
        log(f"   -> dev reduced to {len(dev_rec):,} docs that are NOT test cases (used for selection + threshold tuning)")
    tr_rec = _records("train")
    if SMOKE:
        pick_ = lambda recs, n: [recs[i] for i in rng.choice(len(recs), min(n, len(recs)), replace=False)]
        tr_rec, dev_rec, test_rec = pick_(tr_rec, N_SMOKE["train"]), pick_(dev_rec, N_SMOKE["dev"]), pick_(test_rec, N_SMOKE["test"])
    out = {"train": build_split("train", tr_rec, True), "dev": build_split("dev", dev_rec, True), "test": build_split("test", test_rec, False)}
    del tr_rec, dev_rec, test_rec; gc.collect()
    for s, d in out.items(): save_split(s, d)
    return out

SPLITS = get_all_splits()
ds_train, ds_dev, ds_test = SPLITS["train"], SPLITS["dev"], SPLITS["test"]
assert len(set(ds_test.ids)) == len(ds_test.ids), "duplicate test ids"

# ---------- window geometry ----------
def all_starts(n, W=CHUNK, stride=STRIDE):
    """overlapping window starts; the LAST window is anchored to the document end, so reasoning + disposal are always covered"""
    if n <= W: return np.array([0], dtype=np.int64)
    s = np.arange(0, n - W + 1, stride, dtype=np.int64)
    if s[-1] != n - W: s = np.append(s, n - W)
    return s

def pick(n_win, K, rng=None):
    """coverage-preserving choice of K windows: first + last 2 always, the rest one-per-segment over the middle (random in training, centred at eval)"""
    if n_win <= K: return np.arange(n_win)
    if K == 1: return np.array([n_win - 1])
    if K == 2: return np.array([0, n_win - 1])
    keep = {0, n_win - 2, n_win - 1}
    m = K - 3
    if m > 0:
        lo, hi = 1, n_win - 3
        edges = np.linspace(lo, hi + 1, m + 1)
        for i in range(m):
            a = int(np.ceil(edges[i])); b = max(int(np.ceil(edges[i + 1])), a + 1)
            keep.add(min(int(rng.integers(a, b)) if rng is not None else (a + b - 1) // 2, hi))
    return np.array(sorted(keep))

def bucket_ids(g, n_all):
    """position-embedding id of a window: 0 = first, NB+1 = last, 1..NB = relative position in between"""
    g = np.asarray(g)
    if n_all <= 1: return np.zeros(len(g), dtype=np.int64)
    b = 1 + np.minimum((g / (n_all - 1) * NB).astype(np.int64), NB - 1)
    b = np.where(g == n_all - 1, NB + 1, b)
    return np.where(g == 0, 0, b).astype(np.int64)

def audit(name, ds):
    nt = ds.n_tok; nw = np.array([len(all_starts(n)) for n in nt])
    log(f"{name:10s} docs={len(ds):6,} | tokens mean={nt.mean():7.0f} median={np.median(nt):6.0f} p90={np.percentile(nt,90):7.0f} p99={np.percentile(nt,99):8.0f} | "
        f"windows/doc mean={nw.mean():4.1f} p95={np.percentile(nw,95):3.0f} | {nw.sum():,} windows total")
    if (ds.labels >= 0).all():
        c = np.bincount(ds.labels, minlength=2)
        log(f"{'':10s} labels: class0={c[0]:,} ({100*c[0]/c.sum():.1f}%)  class1={c[1]:,} ({100*c[1]/c.sum():.1f}%)  imbalance={c.max()/max(c.min(),1):.2f}")
log("=" * 100); audit("train", ds_train); audit("dev(clean)", ds_dev); audit("test", ds_test); log("=" * 100)

cnt = np.bincount(ds_train.labels, minlength=2).astype(np.float64)
CLASS_W = torch.tensor(cnt.sum() / (2.0 * cnt), dtype=torch.float32, device=DEVICE)     # "balanced" inverse-frequency weights
log(f"class weights (train, inverse-frequency): {CLASS_W.tolist()}")

In [ ]:
class AdaptedLinear(nn.Module):
    """frozen base Linear + trainable low-rank update.
    'lorma_plus': LoRMA with additive rank inflation (Bihany et al., ACL Findings 2025, arXiv 2506.07621):
                  W' = (I + (alpha/r) B A) W0   with A: r x d_out, B: d_out x r  (multiplicative on the pretrained weight).
                  Computed in the re-ordered, cheap form  y = y0 + s * B (A y0),  y0 = W0 x   -> same cost as LoRA, nothing d x d is materialised.
                  B = 0 at init -> (I + 0) W0 = W0, i.e. training starts exactly at the pretrained model.
    'lora'      : y = W0 x + s * B (A x)   (additive baseline, only for comparison)"""
    def __init__(self, base, r, alpha, p, mode):
        super().__init__()
        self.base, self.mode, self.scale = base, mode, alpha / r
        a_in = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, a_in)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(p)
    def forward(self, x):
        y = self.base(x)
        z = x if self.mode == "lora" else y
        return y + F.linear(F.linear(self.drop(z), self.A), self.B) * self.scale

def apply_adapters(m):
    n = 0
    for blk in m.blocks:
        for parent in (blk.attn, blk.ffn):
            for name in LORA_TARGETS:
                lin = getattr(parent, name, None)
                if isinstance(lin, nn.Linear):
                    setattr(parent, name, AdaptedLinear(lin, LORA_R, LORA_ALPHA, LORA_DROPOUT, ADAPTER)); n += 1
    return n

n_adapted = apply_adapters(model)
model.to(DEVICE)
FEAT_DIM = 2 * MCFG.d_model
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"Model-3: {n_total - n_train:,} frozen params | {n_adapted} adapted Linear layers ({ADAPTER}, r={LORA_R}) -> {n_train:,} trainable ({100*n_train/n_total:.2f}%)")

def backbone_features(m, ids, lens):
    """ids (N,T) right-padded, lens (N,). Causal attention => padding never influences real tokens, so no mask is needed.
    Per-window feature = [state of the last real token ; mean over real tokens] of the (normalised) next-token head-0 output."""
    T = ids.size(1)
    h = m.heads[0](m.hidden(ids)).float()
    ar = torch.arange(T, device=ids.device)[None]
    msk = (ar < lens[:, None]).unsqueeze(-1).float()
    last = h[torch.arange(h.size(0), device=ids.device), lens - 1]
    mean = (h * msk).sum(1) / lens[:, None].float()
    return torch.cat([last, mean], dim=-1)

class DocHead(nn.Module):
    """window features -> doc logits. ATTENTION-BASED POOLING ACROSS CHUNKS: gated-attention multiple-instance pooling (Ilse et al. 2018)
    + learned chunk-position embeddings (first / relative position / last) + explicit tail-chunk branch + per-chunk auxiliary classifier."""
    def __init__(self, in_dim, d=256, att=128, p=0.15):
        super().__init__()
        self.norm, self.proj = nn.LayerNorm(in_dim), nn.Linear(in_dim, d)
        self.pos = nn.Embedding(NB + 2, d)
        self.drop = nn.Dropout(p)
        self.att_v, self.att_u, self.att_w = nn.Linear(d, att), nn.Linear(d, att), nn.Linear(att, 1)
        self.win_cls = nn.Linear(d, 2)
        self.doc_cls = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, 2))
    def forward(self, feats, buckets, mask):
        h = F.gelu(self.proj(self.drop(self.norm(feats)))) + self.pos(buckets)
        h = self.drop(h)
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).squeeze(-1)
        a = a.masked_fill(~mask, -1e4)
        w = torch.softmax(a, dim=-1) * mask
        pooled = (w.unsqueeze(-1) * h).sum(1)
        last_idx = mask.long().sum(1) - 1
        h_last = h[torch.arange(h.size(0), device=h.device), last_idx]      # windows are ordered => last valid = tail window
        return self.doc_cls(torch.cat([pooled, h_last], dim=-1)), self.win_cls(h), w

head = DocHead(FEAT_DIM).to(DEVICE)
log(f"attention-pooling head params: {sum(p.numel() for p in head.parameters()):,}")

def amp_sanity():
    """autocast vs fp32 on real windows - falls back to fp32 if they disagree"""
    global USE_AMP
    if not USE_AMP: return
    seqs = [ds_train.slice(i, 0, min(CHUNK, ds_train.n_tok[i])) for i in range(min(6, len(ds_train)))]
    T = max(len(s) for s in seqs); ids = np.zeros((len(seqs), T), dtype=np.int64); lens = np.array([len(s) for s in seqs])
    for i, s in enumerate(seqs): ids[i, :len(s)] = s
    ids, lens = torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lens).to(DEVICE)
    model.eval()
    with torch.no_grad():
        f32 = backbone_features(model, ids, lens)
        with amp_ctx(): f16 = backbone_features(model, ids, lens)
    cos_min = F.cosine_similarity(f32, f16.float(), dim=-1).min().item()
    ok = bool(torch.isfinite(f16).all()) and cos_min > 0.98
    log(f"AMP sanity: min cosine(fp32, {AMP_DTYPE})={cos_min:.5f} finite={bool(torch.isfinite(f16).all())} -> {'autocast ON' if ok else 'DISABLED (fp32)'}")
    USE_AMP = ok
amp_sanity()

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score, confusion_matrix

def pad_windows(seqs):
    T = max(len(s) for s in seqs)
    arr = np.zeros((len(seqs), T), dtype=np.int64)
    for i, s in enumerate(seqs): arr[i, :len(s)] = s
    return arr, np.array([len(s) for s in seqs], dtype=np.int64)

def make_train_batch(ds, idxs, rng):
    seqs, doc_of, slot_of, rows, ys = [], [], [], [], []
    for b, di in enumerate(idxs):
        n = int(ds.n_tok[di]); starts = all_starts(n); nw = len(starts)
        sel = pick(nw, K_TRAIN, rng)                                 # random middle windows each epoch = augmentation for long docs
        for slot, g in enumerate(sel):
            s = int(starts[g]); seqs.append(ds.slice(di, s, min(CHUNK, n - s))); doc_of.append(b); slot_of.append(slot)
        rows.append(bucket_ids(sel, nw)); ys.append(int(ds.labels[di]))
    ids, lens = pad_windows(seqs)
    B, Km = len(idxs), max(len(r) for r in rows)
    bucket = np.zeros((B, Km), dtype=np.int64); mask = np.zeros((B, Km), dtype=bool)
    for i, r in enumerate(rows): bucket[i, :len(r)] = r; mask[i, :len(r)] = True
    t = lambda a: torch.from_numpy(a).to(DEVICE, non_blocking=True)
    return dict(ids=t(ids), lens=t(lens), doc_of=t(np.array(doc_of)), slot_of=t(np.array(slot_of)),
                bucket=t(bucket), mask=t(mask), y=t(np.array(ys, dtype=np.int64)))

@torch.no_grad()
def extract_features(ds, doc_indices, kcap, bs=None, tag=""):
    """run the LoRMA-adapted backbone over the selected overlapping windows of every doc; returns window features + bookkeeping,
    so any doc-level aggregator / section ablation can be evaluated later without touching the GPU again"""
    bs = bs or INFER_BS
    was_training = model.training
    model.eval(); head.eval()
    items, ptr, gidx, nall = [], [0], [], []
    for di in doc_indices:
        n = int(ds.n_tok[di]); starts = all_starts(n); nw = len(starts)
        for g in pick(nw, kcap):
            s = int(starts[g]); items.append((int(di), s, min(CHUNK, n - s))); gidx.append(int(g))
        ptr.append(len(items)); nall.append(nw)
    feats = np.zeros((len(items), FEAT_DIM), dtype=np.float32)
    t0 = time.time(); nb = (len(items) + bs - 1) // bs
    for bi in range(nb):
        chunk = items[bi * bs:(bi + 1) * bs]
        ids, lens = pad_windows([ds.slice(di, s, l) for di, s, l in chunk])
        with amp_ctx():
            f = backbone_features(model, torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lens).to(DEVICE))
        feats[bi * bs: bi * bs + len(chunk)] = f.float().cpu().numpy()
        if tag and (bi % 100 == 0 or bi == nb - 1):
            log(f"    [{tag}] batch {bi+1}/{nb} | {len(items)/max(time.time()-t0,1e-6):.0f} windows/s (avg)" if bi else f"    [{tag}] {len(items):,} windows from {len(doc_indices):,} docs")
    if was_training: model.train(); head.train()
    return feats, np.array(ptr), np.array(gidx), np.array(nall)

@torch.no_grad()
def doc_probs(feats, ptr, gidx, nall, keep=None, agg="att", collect_att=False):
    """P(class=1) per document from cached window features.
    keep(g, n_all)->bool mask restricts WHICH windows the head may see (section ablations); agg='winmean' = mean of window logits (no attention)"""
    head.eval()
    ND = len(ptr) - 1; probs = np.zeros(ND, dtype=np.float64); att = []
    for b0 in range(0, ND, DOC_BS):
        docs = list(range(b0, min(ND, b0 + DOC_BS))); rows = []
        for d in docs:
            sel = np.arange(ptr[d], ptr[d + 1])
            if keep is not None:
                m = np.asarray(keep(gidx[sel], int(nall[d])), dtype=bool)
                sel = sel[m] if m.any() else sel[-1:]
            rows.append(sel)
        Km = max(len(r) for r in rows)
        Fb = np.zeros((len(docs), Km, FEAT_DIM), dtype=np.float32); Bk = np.zeros((len(docs), Km), dtype=np.int64); Mk = np.zeros((len(docs), Km), dtype=bool)
        for i, (d, r) in enumerate(zip(docs, rows)):
            Fb[i, :len(r)] = feats[r]; Bk[i, :len(r)] = bucket_ids(gidx[r], int(nall[d])); Mk[i, :len(r)] = True
        Ft, Bt, Mt = (torch.from_numpy(a).to(DEVICE) for a in (Fb, Bk, Mk))
        logits, wlog, w = head(Ft, Bt, Mt)
        if agg == "winmean":
            logits = (wlog * Mt.unsqueeze(-1)).sum(1) / Mt.sum(1, keepdim=True)
        probs[b0:b0 + len(docs)] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
        if collect_att:
            wn = w.cpu().numpy()
            for i, r in enumerate(rows): att.append((Bk[i, :len(r)], wn[i, :len(r)]))
    return (probs, att) if collect_att else probs

def macro_f1_np(y, pred):
    f = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
        f.append(2 * p * r / (p + r) if p + r else 0.0)
    return float(np.mean(f))

def tune_threshold(y, p, lo=0.20, hi=0.80, step=0.01, smooth=5):
    """threshold maximising Macro-F1 on a smoothed F1 curve (so one lucky threshold on a small dev set is not chosen)"""
    grid = np.arange(lo, hi + 1e-9, step)
    f = np.array([macro_f1_np(y, (p >= t).astype(int)) for t in grid])
    fs = np.convolve(np.pad(f, smooth // 2, mode="edge"), np.ones(smooth) / smooth, mode="valid")
    i = int(np.argmax(fs)); return float(grid[i]), float(f[i]), float(macro_f1_np(y, (p >= 0.5).astype(int)))

def full_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    P, R, Fm, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    pc, rc, fc, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    try: auc = roc_auc_score(y, p)
    except ValueError: auc = float("nan")
    return dict(thr=thr, macro_f1=Fm, macro_precision=P, macro_recall=R, accuracy=accuracy_score(y, pred), auc=auc,
                p0=pc[0], r0=rc[0], f1_0=fc[0], p1=pc[1], r1=rc[1], f1_1=fc[1], confusion=confusion_matrix(y, pred, labels=[0, 1]).tolist())

_rs = np.random.default_rng(SEED + 1)
QUICK_IDX = np.sort(_rs.choice(len(ds_dev), min(N_QUICK, len(ds_dev)), replace=False))

def quick_eval():
    was_training = model.training
    f, ptr, g, na = extract_features(ds_dev, QUICK_IDX, K_QUICK)
    p = doc_probs(f, ptr, g, na); y = ds_dev.labels[QUICK_IDX]
    thr, f1_tuned, f1_05 = tune_threshold(y, p)
    try: auc = roc_auc_score(y, p)
    except ValueError: auc = float("nan")
    if was_training: model.train(); head.train()
    return dict(auc=auc, f1_05=f1_05, f1_tuned=f1_tuned, thr=thr)

In [ ]:
trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW([{"params": trainable, "lr": LR_LORA, "weight_decay": 0.0},
                         {"params": list(head.parameters()), "lr": LR_HEAD, "weight_decay": WD_HEAD}], betas=(0.9, 0.999))
BASE_LRS = [LR_LORA, LR_HEAD]
_use_scaler = USE_AMP and AMP_DTYPE == torch.float16
try:    scaler = torch.amp.GradScaler("cuda", enabled=_use_scaler)
except Exception: scaler = torch.cuda.amp.GradScaler(enabled=_use_scaler)

S = dict(epoch=0, doc_ptr=0, step=0, train_sec=0.0, docs=0, wins=0, best_auc=-1.0, best_step=-1)
LATEST, BEST = CKPT / "latest_cjpe_m3.pt", CKPT / "best_adapter_m3.pt"
RUN_CFG = dict(model="model3_mtp", adapter=ADAPTER, r=LORA_R, alpha=LORA_ALPHA, chunk=CHUNK, stride=STRIDE, k_train=K_TRAIN, lowercase=LOWERCASE,
               feat_dim=FEAT_DIM, targets=list(LORA_TARGETS), tokenizer=TOK.name, tok_fp=TOK.fingerprint)
_compat = lambda c: c.get("model") == "model3_mtp" and c.get("adapter") == ADAPTER and c.get("feat_dim") == FEAT_DIM and c.get("tok_fp") == TOK.fingerprint

def adapter_state():
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

def load_adapter_state(lora_sd, head_sd):
    own = dict(model.named_parameters()); miss = 0
    for n, v in lora_sd.items():
        if n in own: own[n].data.copy_(v.to(own[n].device))
        else: miss += 1
    head.load_state_dict(head_sd)
    if miss: log(f"warning: {miss} adapter tensors in the checkpoint did not match this model")

def save_latest():
    atomic_save({"lora": adapter_state(), "head": head.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "state": dict(S), "cfg": RUN_CFG}, LATEST)

def save_best(res):
    atomic_save({"lora": adapter_state(), "head": head.state_dict(), "cfg": RUN_CFG, "quick": res, "step": S["step"]}, BEST)

def try_resume():
    if SMOKE: return False
    cands = [p for p in [str(LATEST)] + find_files("latest_cjpe_m3.pt", [INPUT_ROOT]) if os.path.exists(p)]
    best = None
    for p in cands:
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if _compat(ck.get("cfg", {})) and (best is None or ck["state"]["train_sec"] > best[1]["state"]["train_sec"]): best = (p, ck)
        except Exception as e:
            log(f"could not read {p}: {e}")
    if best is None: return False
    p, ck = best
    load_adapter_state(ck["lora"], ck["head"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"]); S.update(ck["state"])
    log(f"RESUMED from {p}: step {S['step']}, epoch {S['epoch']}, {S['train_sec']/60:.0f} min already trained, best quick AUC {S['best_auc']:.4f}")
    prev_best = os.path.join(os.path.dirname(p), "best_adapter_m3.pt")
    if not BEST.exists() and os.path.exists(prev_best): shutil.copy(prev_best, BEST)
    return True

def lr_mult(step, progress):
    warm = min(1.0, (step + 1) / WARMUP_STEPS)
    return warm * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0))))

def micro_step(idxs, rng):
    b = make_train_batch(ds_train, idxs, rng)
    with amp_ctx():
        feats = backbone_features(model, b["ids"], b["lens"])
    feats = feats.float()
    B, Km = b["mask"].shape
    padded = torch.zeros(B, Km, feats.size(-1), device=feats.device).index_put((b["doc_of"], b["slot_of"]), feats)
    logits, wlog, _ = head(padded, b["bucket"], b["mask"])
    loss_doc = F.cross_entropy(logits, b["y"], weight=CLASS_W)                                  # class imbalance: inverse-frequency weights
    wy = b["y"][:, None].expand(B, Km)[b["mask"]]
    loss_aux = F.cross_entropy(wlog[b["mask"]], wy, weight=CLASS_W, label_smoothing=0.1)        # every chunk sees the doc label (noisy -> smoothed)
    loss = loss_doc + AUX_W * loss_aux
    if not torch.isfinite(loss): return float("nan"), 0
    scaler.scale(loss / ACCUM).backward()
    return float(loss.item()), int(b["ids"].size(0))

def train():
    N = len(ds_train); budget = TRAIN_BUDGET_MIN * 60.0
    session_cap = max(0.0, (SESSION_LIMIT_H * 60 - RESERVE_MIN) * 60 - (time.time() - T0))
    t_sess, sec0 = time.time(), S["train_sec"]
    last_ckpt = last_eval = t_log = time.time(); w_log, losses, n_oom, stop, docs_sess = 0, [], 0, None, 0
    model.train(); head.train()
    log(f"training: budget {TRAIN_BUDGET_MIN} min total ({sec0/60:.0f} done) | this session may use {session_cap/60:.0f} min | "
        f"{MICRO_DOCS*ACCUM} docs/step x {K_TRAIN} windows | {N:,} train docs")
    while stop is None and S["epoch"] < MAX_EPOCHS:
        perm = np.random.default_rng(SEED + S["epoch"]).permutation(N)
        rng = np.random.default_rng(SEED * 7 + S["epoch"] * 1000 + S["step"])
        while S["doc_ptr"] < N:
            sess = time.time() - t_sess; elapsed = sec0 + sess; S["train_sec"] = elapsed
            progress = max(elapsed / budget, (S["epoch"] * N + S["doc_ptr"]) / (MAX_EPOCHS * N))
            if progress >= 1.0: stop = "budget reached"; break
            if sess >= session_cap: stop = "session limit reached - resume next session"; break
            m = lr_mult(S["step"], progress)
            for g, base in zip(opt.param_groups, BASE_LRS): g["lr"] = base * m
            opt.zero_grad(set_to_none=True); nmb = 0
            for _ in range(ACCUM):
                if S["doc_ptr"] >= N: break
                idxs = perm[S["doc_ptr"]: S["doc_ptr"] + MICRO_DOCS]; S["doc_ptr"] += len(idxs)
                try:
                    l, nw = micro_step(idxs, rng)
                except OOM_ERR:
                    n_oom += 1; opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                    log(f"  CUDA OOM #{n_oom} - skipped a micro-batch (lower MICRO_DOCS or K_TRAIN if this repeats)")
                    if n_oom > 5: raise
                    continue
                if nw: losses.append(l); w_log += nw; S["docs"] += len(idxs); S["wins"] += nw; docs_sess += len(idxs); nmb += 1
            if nmb == 0: continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable + list(head.parameters()), GRAD_CLIP)
            scaler.step(opt); scaler.update(); S["step"] += 1
            if S["step"] % LOG_EVERY == 0:
                dt = time.time() - t_log; wps = w_log / max(dt, 1e-6)
                eta_epoch = (N - S["doc_ptr"]) / max(docs_sess / max(sess, 1e-6), 1e-6)
                log(f"ep {S['epoch']} step {S['step']:5d} | docs {S['doc_ptr']:6,}/{N:,} | loss {np.mean(losses[-LOG_EVERY*ACCUM:]):.4f} | "
                    f"lr x{m:.2f} | {wps:5.1f} win/s | trained {fmt_t(elapsed)}/{fmt_t(budget)} | epoch ETA {fmt_t(eta_epoch)}")
                t_log, w_log = time.time(), 0
            now = time.time()
            if now - last_eval >= EVAL_EVERY_MIN * 60:
                r = quick_eval(); last_eval = time.time()
                better = (not np.isnan(r["auc"])) and r["auc"] > S["best_auc"]
                if better: S["best_auc"], S["best_step"] = r["auc"], S["step"]; save_best(r)
                log(f"  >> quick-dev ({len(QUICK_IDX)} docs, {K_QUICK} windows): AUC {r['auc']:.4f} | macroF1@0.5 {r['f1_05']:.4f} | "
                    f"macroF1@thr{r['thr']:.2f} {r['f1_tuned']:.4f} {'(NEW BEST)' if better else ''}")
                t_log = time.time()
            if now - last_ckpt >= CKPT_EVERY_MIN * 60: save_latest(); last_ckpt = now
        if stop is None: S["epoch"] += 1; S["doc_ptr"] = 0
    S["train_sec"] = sec0 + (time.time() - t_sess)
    r = quick_eval()
    better = (not np.isnan(r["auc"])) and r["auc"] > S["best_auc"]
    if better or not BEST.exists(): S["best_auc"], S["best_step"] = r["auc"], S["step"]; save_best(r)
    log(f"  >> final quick-dev: AUC {r['auc']:.4f} | macroF1@0.5 {r['f1_05']:.4f} | macroF1@thr{r['thr']:.2f} {r['f1_tuned']:.4f} {'(NEW BEST)' if better else ''}")
    save_latest()
    log(f"training stopped: {stop or 'epochs finished'} | steps {S['step']} | docs seen {S['docs']:,} | windows {S['wins']:,} | best quick AUC {S['best_auc']:.4f} @ step {S['best_step']}")

if MODE == "train_predict":
    try_resume()
    train()
else:
    log("MODE=predict_only: skipping training")

In [ ]:
# (dev = dev.json minus every test-case ID, so nothing here touches test labels)
_best = None
for p in [str(BEST)] + find_files("best_adapter_m3.pt", [INPUT_ROOT]):
    if os.path.exists(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if _compat(ck["cfg"]): _best = (p, ck); break
        except Exception as e:
            log(f"could not read {p}: {e}")
if _best:
    load_adapter_state(_best[1]["lora"], _best[1]["head"])
    log(f"evaluating best adapter from {_best[0]} (step {_best[1].get('step')}, quick AUC {_best[1]['quick']['auc']:.4f})")
else:
    assert MODE == "train_predict", "predict_only needs a best_adapter_m3.pt (attach the training run's output as an input)"
    log("no best_adapter_m3.pt found - using the weights currently in memory")

if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
t0 = time.time()
dev_idx = np.arange(len(ds_dev))
f_dev, ptr_dev, g_dev, n_dev = extract_features(ds_dev, dev_idx, MAX_INFER_WINDOWS, tag="dev")
dev_sec = time.time() - t0
log(f"dev feature extraction: {dev_sec/60:.1f} min")
np.savez(OUT / "dev_window_feats_m3.npz", feats=f_dev, ptr=ptr_dev, gidx=g_dev, nall=n_dev, y=ds_dev.labels)
y_dev = ds_dev.labels
p_dev = doc_probs(f_dev, ptr_dev, g_dev, n_dev)

THR, f1_tuned, f1_05 = tune_threshold(y_dev, p_dev)
if abs(THR - 0.5) > 0.15: log(f"warning: tuned threshold {THR:.2f} is far from 0.5 - check for train/dev shift before trusting it")
M05, MT = full_metrics(y_dev, p_dev, 0.5), full_metrics(y_dev, p_dev, THR)
log("\n" + "=" * 90 + f"\nDEV RESULTS ({len(y_dev):,} docs, class balance {np.bincount(y_dev, minlength=2).tolist()})\n" + "=" * 90)
for name, m in (("threshold 0.50", M05), (f"tuned thr {THR:.2f}", MT)):
    log(f"{name:15s} | Macro-F1 {m['macro_f1']:.4f} | Macro-P {m['macro_precision']:.4f} | Macro-R {m['macro_recall']:.4f} | Acc {m['accuracy']:.4f} | AUC {m['auc']:.4f}")
    log(f"{'':15s} | class0 P/R/F1 {m['p0']:.3f}/{m['r0']:.3f}/{m['f1_0']:.3f} | class1 P/R/F1 {m['p1']:.3f}/{m['r1']:.3f}/{m['f1_1']:.3f} | confusion [[TN,FP],[FN,TP]] = {m['confusion']}")

ABL = [("all windows (attention pooling)",     None,                                  "att"),
       ("all windows (mean of window logits)", None,                                  "winmean"),
       ("coverage subset K=6 (training policy)", lambda g, n: np.isin(g, pick(n, 6)), "att"),
       ("first window only",                   lambda g, n: g == 0,                   "att"),
       ("last window only",                    lambda g, n: g == n - 1,               "att"),
       ("last 2 windows",                      lambda g, n: g >= n - 2,               "att"),
       ("first + last 2 windows",              lambda g, n: (g == 0) | (g >= n - 2),  "att"),
       ("first half of document",              lambda g, n: g <= (n - 1) / 2,         "att"),
       ("second half of document",             lambda g, n: g >= (n - 1) / 2,         "att")]
rows = []
for name, keep, agg in ABL:
    p = p_dev if (keep is None and agg == "att") else doc_probs(f_dev, ptr_dev, g_dev, n_dev, keep=keep, agg=agg)
    thr_, f_t, f_05 = tune_threshold(y_dev, p)
    rows.append(dict(setting=name, AUC=roc_auc_score(y_dev, p), MacroF1_at_0_5=f_05, MacroF1_tuned_thr=f_t, thr=thr_))
abl_df = pd.DataFrame(rows).round(4)
log("\nSECTION / POOLING ABLATION (same trained model, restricted input; tuned-thr column is optimistic - tuned on dev itself)")
log(abl_df.to_string(index=False)); abl_df.to_csv(OUT / "ablation_sections_m3.csv", index=False)

_, att = doc_probs(f_dev, ptr_dev, g_dev, n_dev, collect_att=True)
mass = np.zeros(NB + 2); cnt_ = np.zeros(NB + 2)
for b, w in att:
    for bi, wi in zip(b, w): mass[bi] += wi; cnt_[bi] += 1
labels_b = ["first"] + [f"rel {i/NB:.2f}-{(i+1)/NB:.2f}" for i in range(NB)] + ["last"]
att_df = pd.DataFrame({"window position": labels_b, "mean attention weight": mass / np.maximum(cnt_, 1), "share of total attention": mass / mass.sum(), "n windows": cnt_.astype(int)}).round(4)
log("\nATTENTION BY WINDOW POSITION (dev)"); log(att_df.to_string(index=False)); att_df.to_csv(OUT / "attention_by_position_m3.csv", index=False)

pred_dev = (p_dev >= THR).astype(int); ntok = ds_dev.n_tok
grp = np.where((pred_dev == 1) & (y_dev == 1), "TP", np.where((pred_dev == 0) & (y_dev == 0), "TN", np.where((pred_dev == 1) & (y_dev == 0), "FP", "FN")))
err_df = pd.DataFrame({"group": grp, "tokens": ntok, "windows": n_dev, "p1": p_dev, "mixed_case_src": ds_dev.has_upper})
summ = err_df.groupby("group").agg(n=("tokens", "size"), mean_tokens=("tokens", "mean"), median_tokens=("tokens", "median"), mean_windows=("windows", "mean"), mean_p1=("p1", "mean"), mixed_case_share=("mixed_case_src", "mean")).round(2)
log("\nERROR ANALYSIS at tuned threshold (FP = predicted 1, true 0 | FN = predicted 0, true 1)"); log(summ.to_string())
q = pd.qcut(ntok, 4, duplicates="drop")
er = pd.DataFrame({"len_quartile": q, "wrong": pred_dev != y_dev, "y": y_dev}).groupby("len_quartile", observed=True).agg(docs=("wrong", "size"), error_rate=("wrong", "mean"), class1_share=("y", "mean")).round(3)
log("\nERROR RATE BY DOCUMENT LENGTH QUARTILE (tokens)"); log(er.to_string())
summ.to_csv(OUT / "error_groups_m3.csv"); er.to_csv(OUT / "error_by_length_m3.csv")
pd.DataFrame({"id": ds_dev.ids, "label": y_dev, "p1": p_dev, "pred": pred_dev, "tokens": ntok, "windows": n_dev}).to_csv(OUT / "dev_probs_m3.csv", index=False)

# ---------- efficiency (model size / memory / inference cost) ----------
n_head = sum(p.numel() for p in head.parameters())
eff = dict(backbone_frozen_params=n_total - n_train, lorma_trainable_params=n_train, head_params=n_head,
           trainable_pct=100 * n_train / n_total, adapter_head_ckpt_MB=(os.path.getsize(BEST) / 2**20 if BEST.exists() else None),
           backbone_fp16_MB=2 * (n_total - n_train) / 2**20, peak_gpu_GB=(torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else None),
           dev_windows=int(len(f_dev)), dev_windows_per_s=len(f_dev) / max(dev_sec, 1e-6), tokenizer=TOK.name)
log("\nEFFICIENCY: " + json.dumps(eff, default=float))
json.dump({"threshold": THR, "dev_at_0.5": M05, "dev_at_tuned": MT, "run": RUN_CFG, "train_state": S, "efficiency": eff, "tokenizer_probe_loss": probe_loss},
          open(OUT / "dev_metrics_m3.json", "w"), indent=1, default=float)

In [ ]:
t0 = time.time()
test_idx = np.arange(len(ds_test))
f_te, ptr_te, g_te, n_te = extract_features(ds_test, test_idx, MAX_INFER_WINDOWS, tag="test")
log(f"test feature extraction: {(time.time()-t0)/60:.1f} min")
np.savez(OUT / "test_window_feats_m3.npz", feats=f_te, ptr=ptr_te, gidx=g_te, nall=n_te)
p_te = doc_probs(f_te, ptr_te, g_te, n_te)
pred_te = (p_te >= THR).astype(int)

pd.DataFrame({"id": ds_test.ids, "p1": p_te, "pred": pred_te}).to_csv(OUT / "test_probs_m3.csv", index=False)      # probabilities kept for ensembling
submission = [{"id": i, "pred": int(v)} for i, v in zip(ds_test.ids, pred_te)]

assert len(submission) == len(ds_test) == len(set(ds_test.ids)), "one prediction per test case required"
assert all(set(s) == {"id", "pred"} and type(s["pred"]) is int and s["pred"] in (0, 1) for s in submission)
if not SMOKE:
    ref = [str(r["id"]) for r in _records("test")]
    assert [s["id"] for s in submission] == ref, "submission ids/order differ from test.json"
sub_path = SUB / "prediction.json"
with open(sub_path, "w", encoding="utf-8") as f: json.dump(submission, f)
with zipfile.ZipFile(SUB / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(sub_path, "prediction.json")

log("\n" + "=" * 90)
log(f"SUBMISSION WRITTEN: {sub_path}   ({len(submission):,} predictions{' - SMOKE RUN, NOT A VALID SUBMISSION' if SMOKE else ''})")
log(f"predicted class balance: class0={int((pred_te==0).sum()):,} ({100*(pred_te==0).mean():.1f}%) | class1={int((pred_te==1).sum()):,} ({100*(pred_te==1).mean():.1f}%) | threshold {THR:.2f}")
log(f"dev Macro-F1 (leak-free dev, tuned thr): {MT['macro_f1']:.4f} | at 0.5: {M05['macro_f1']:.4f} | AUC {MT['auc']:.4f}")
log(f"total wall time: {(time.time()-T0)/60:.0f} min | outputs in {WORK}")
log("=" * 90)